# Kuantisasi (Quantization) & QLoRA
Materi ini mencakup bagaimana Anda dapat mengemat penggunaan VRAM secara drastis saat memuat dan melakukan *fine-tuning* pada model besar (seperti BERT, LLaMA, GPT) menggunakan teknik **Kuantisasi** dan algoritma **LoRA (Low-Rank Adaptation)**.

Saat digabungkan, teknik ini disebut **QLoRA**.

> **Note:** Anda mungkin memerlukan instalasi *library* tambahan jika Anda menggunakan environment lokal, Anda bisa jalankan:
> `!pip install bitsandbytes peft accelerate`

In [2]:
import torch
from transformers import AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_name = "bert-base-uncased"
print(f"Device aktif: {device}")

Device aktif: cuda


## 1. Modul 4-Bit BitsAndBytes (PTQ)

Secara default, bobot transformer termuat dalam ukuran `32-bit floating point (FP32)`. Menggunakan `BitsAndBytesConfig`, kita memuat secara langsung model ke ukuran komputasi `4-bit (NF4)`. Hal ini merampingkan memori asalnya menjadi ~1/8 dari ukuran semula. Biasa disebut Post-Training Quantization (PTQ).

In [3]:
model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
for p in model.parameters():
    print(p)
    break

Parameter containing:
tensor([[-0.0102, -0.0615, -0.0265,  ..., -0.0199, -0.0372, -0.0098],
        [-0.0117, -0.0600, -0.0323,  ..., -0.0168, -0.0401, -0.0107],
        [-0.0198, -0.0627, -0.0326,  ..., -0.0165, -0.0420, -0.0032],
        ...,
        [-0.0218, -0.0556, -0.0135,  ..., -0.0043, -0.0151, -0.0249],
        [-0.0462, -0.0565, -0.0019,  ...,  0.0157, -0.0139, -0.0095],
        [ 0.0015, -0.0821, -0.0160,  ..., -0.0081, -0.0475,  0.0753]],
       requires_grad=True)


In [7]:
print(f"Sukses! Memory Footprint model 4-bit: {model.get_memory_footprint() / 1e6:.2f} MB")

Sukses! Memory Footprint model 4-bit: 437.94 MB


In [5]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4'
)

quatizatized_model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2,
    quantization_config=bnb_config
)

W0522 11:45:29.816000 14312 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

c:\Users\Dindin\.pyenv\pyenv-win\versions\3.10.11\lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Note

In [6]:
for p in quatizatized_model.parameters():
    print(p)
    break

Parameter containing:
tensor([[-0.0102, -0.0615, -0.0265,  ..., -0.0199, -0.0372, -0.0098],
        [-0.0117, -0.0600, -0.0323,  ..., -0.0168, -0.0401, -0.0107],
        [-0.0198, -0.0627, -0.0326,  ..., -0.0165, -0.0420, -0.0032],
        ...,
        [-0.0218, -0.0556, -0.0135,  ..., -0.0043, -0.0151, -0.0249],
        [-0.0462, -0.0565, -0.0019,  ...,  0.0157, -0.0139, -0.0095],
        [ 0.0015, -0.0821, -0.0160,  ..., -0.0081, -0.0475,  0.0753]],
       device='cuda:0', requires_grad=True)


In [9]:
print(f"Sukses! Memory Footprint model 4-bit: {quatizatized_model.get_memory_footprint() / 1e6:.2f} MB")

Sukses! Memory Footprint model 4-bit: 138.61 MB


## 2. Praktik QLoRA (Quantized LoRA) dengan PEFT

Karena parameter internal dari model kuantisasi 4-bit sudah dikonversi (dan terkunci / tidak bisa *compute gradients* dalam nilai utuh), maka model dasar **TIDAK BISA DI-TRAINING ULANG LANGSUNG**.

Sebagai solusinya:
1. Kita bekukan parameter aslinya.
2. Kita tempelkan *"sayap kecil"* berukuran mungil (adapters) ke layer Attention. Sayap inilah dinamakan **LoRA (Low-Rank Adaptation)**. Bagian *sayap* inilah yang nantinya dilatih (fine-tune) terhadap dataset Anda menggunakan parameter utuh / presisi tinggi.

In [12]:
quantized_model = prepare_model_for_kbit_training(quatizatized_model)

lora_config = LoraConfig(
    task_type='SEQ_CLS',
    target_modules=['key','value'],
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none'
)

peft_model = get_peft_model(model, lora_config)
peft_model

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): BertForSequenceClassification(
      (bert): BertModel(
        (embeddings): BertEmbeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (token_type_embeddings): Embedding(2, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): BertEncoder(
          (layer): ModuleList(
            (0-11): 12 x BertLayer(
              (attention): BertAttention(
                (self): BertSelfAttention(
                  (query): Linear(in_features=768, out_features=768, bias=True)
                  (key): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                   